In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

In [0]:
import os
import re
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit


In [0]:
spark = SparkSession.builder \
    .appName("LeituraXLSXComData") \
    .config("spark.jars.packages", "com.crealytics:spark-excel_2.12:0.13.5") \
    .getOrCreate()

In [0]:
diretorio = r'/Volumes/databox/juridico_comum/arquivos/regulatório/bases_fechamento_financeiro/'
arquivos = [os.path.join(diretorio, f) for f in os.listdir(diretorio) if f.endswith(".xlsx")]


In [0]:
dfs = []

colunas_desejadas = [
'LINHA',
'ID PROCESSO',
'PASTA',
'Área do Direito',
'Sub-área do Direito',
'Grupo (M-1)',
'Empresa (M-1)',
'Grupo (M)',
'Empresa (M)',
'STATUS (M-1)',
'STATUS (M)',
'Centro de Custo (M-1)',
'Centro de Custo (M)',
'DATACADASTRO',
'Reabertura',
'Objeto Assunto/Cargo (M-1)',
'Sub Objeto Assunto/Cargo (M-1)',
'Objeto Assunto/Cargo (M)',
'Sub Objeto Assunto/Cargo (M)',
'Órgão ofensor (Fluxo) (M-1)',
'Órgão ofensor (Fluxo) (M)',
'Natureza Operacional (M-1)',
'Natureza Operacional (M)',
'Média de Pagamento',
'% Risco',
'Distribuição',
'Nº Processo',
'Escritorio',
'Vlr Causa',
'% Sócio (M-1)',
'% Empresa (M-1)',
'% Sócio (M)',
'% Empresa (M)',
'Provisão (M-1)',
'Correção  (M-1)',
'Provisão Total (M-1)',
'Classificação Mov (M)',
'Provisão Mov  (M)',
'Correção Mov  (M)',
'Provisão Mov  Total (M)',
'Provisão Total (M)',
'Correção (M-1)',
'Correção (M)',
'Provisão Total Passivo (M)',
'Socio: Provisão (M-1)',
'Socio: Correção (M-1)',
'Socio: Provisão Total (M-1)',
'SOCIO: Classificação Mov  (M)',
'SOCIO: Provisão Mov  (M)',
'SOCIO: Correção Mov  (M)',
'SOCIO: Provisão Mov  Total (M)',
'SOCIO: Provisão Total (M)',
'SOCIO: Correção (M-1)_0001',
'SOCIO: Correção (M)',
'SOCIO: Prov  Total Passivo (M)',
'Empresa: Provisão (M-1)',
'Empresa: Correção (M-1)',
'Empresa: Provisão Total (M-1)',
'EMPRESA: Classificação Mov.(M)',
'EMPRESA: Provisão Mov  (M)',
'EMPRESA: Correção Mov  (M)',
'EMPRESA: Provisão Mov Total(M)',
'EMPRESA: Provisão Total (M)',
'EMPRESA: Correção (M-1)_0001',
'EMPRESA: Correção (M)',
'EMPRESA: Prov Total Passivo(M)',
'Demitido por Reestruturação',
'Indicação Processo Estratégico',
'Filial (M)',
'NOVOS',
'ENCERRADOS',
'ESTOQUE',
'SUB TIPO',
'TIPO_PGTO',
'SUM_of_VALOR',
'OUTRAS_ADICOES',
'OUTRAS_REVERSOES',
'Indicação Processo Estratégico']


In [0]:
for caminho in arquivos:
    nome_arquivo = os.path.basename(caminho)

    # Extrai data do nome do arquivo, ex: "dados_2024-03.xlsx" → "2024-03"
    match = re.search(r"(\d{4}-\d{2})", nome_arquivo)
    data_fec = match.group(1) if match else "desconhecido"

    # Lê o arquivo Excel
    df = spark.read.format("com.crealytics.spark.excel") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("dataAddress", "'Base'!A1") \
        .load(caminho)

    # Filtra colunas e adiciona colunas extras
    df_filtrado = df.select(*colunas_desejadas) \
        .withColumn("data_fec", lit(data_fec)) \
        .withColumn("origem_arquivo", lit(nome_arquivo))

    dfs.append(df_filtrado)

# Concatena todos
df_final = dfs[0]
for df in dfs[1:]:
    df_final = df_final.unionByName(df)

# Exibe resultado
df_final.show()
